# 01 — GEDI preprocessing and catalogue audit

This notebook audits the retained GEDI L2A RH95 catalogue produced by the preprocessing workflow. It verifies quality-control outputs, unique-shot identifiers, spatial partitions, and the reference-height support used by the manuscript. Raw GEDI granules are not distributed with the repository.

In [ ]:
from pathlib import Path
import hashlib
import pandas as pd
ROOT = Path.cwd().resolve()
CATALOG_ROOT = ROOT / 'data' / 'processed' / 'gedi'
CATALOGS = {site: CATALOG_ROOT/site/'shot_catalog_step05.csv.gz' for site in ('ifran','maamoura','agadir')}
CATALOGS


## Publication audit summary

The checks below export compact tables used to document the GEDI support of the three study landscapes.

In [ ]:
def digest(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8*1024*1024), b''):
            h.update(block)
    return h.hexdigest()

rows = []
for site, path in CATALOGS.items():
    frame = pd.read_csv(path, low_memory=False)
    required = {'split', 'aux_shot_uid', 'rh95'}
    assert required.issubset(frame.columns), (site, required - set(frame.columns))
    assert frame['split'].isin(['train','val','test']).all()
    unique = frame.drop_duplicates(['split','aux_shot_uid'])
    counts = unique['split'].value_counts()
    rows.append({'site': site, 'occurrence_rows': len(frame), 'unique_shots': len(unique), 'train': int(counts.get('train',0)), 'val': int(counts.get('val',0)), 'test': int(counts.get('test',0)), 'rh95_min_m': unique.rh95.min(), 'rh95_max_m': unique.rh95.max(), 'sha256': digest(path)})
audit = pd.DataFrame(rows)
display(audit)
